# Populations, Communities, and Ecosystem Dynamics Workflow

This notebook scaffold supports the article **Populations, Communities, and Ecosystem Dynamics**. It is intentionally minimal and can be expanded with community turnover, ecosystem reorganization risk, trophic simulations, and provenance documentation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform

article_dir = Path.cwd().parent
community = pd.read_csv(article_dir / 'data' / 'community_matrix.csv').set_index('site')
indicators = pd.read_csv(article_dir / 'data' / 'ecosystem_indicators.csv').set_index('site')
community.head()

In [ ]:
relative_abundance = community.div(community.sum(axis=1), axis=0)
safe_relative_abundance = relative_abundance.replace(0, np.nan)
shannon = -(
    safe_relative_abundance * np.log(safe_relative_abundance)
).sum(axis=1).fillna(0)

bray_curtis = squareform(pdist(community.values, metric='braycurtis'))
bray_curtis_df = pd.DataFrame(bray_curtis, index=community.index, columns=community.index)

risk = pd.DataFrame(index=community.index)
risk['richness'] = (community > 0).sum(axis=1)
risk['shannon'] = shannon
risk['mean_turnover'] = bray_curtis_df.mean(axis=1)
risk = risk.join(indicators)
risk.round(3)